# Supply-chain analyses

This notebook shows one end-to-end workflow for using MARIO to study a supply chain with EXIOBASE. Starting from yearly parser inputs, the same `Database` instance is used to move from structural indicators to bilateral trade matrices, embodied trade content, contributor breakdowns, concentration, and exposure.

The example focuses on `Nickel ores and concentrates`, but the same pattern can be reused for any sector or commodity.

In [23]:
import mario

## 1. Define the EXIOBASE inputs and the focal supply chain

The workflow below assumes that you have a local folder with yearly EXIOBASE archives. The first year initializes the MARIO database, and the remaining years are imported as additional scenarios so that the same supply chain can be compared through time.

In [ ]:
FOLDER = 'YOUR_PATH'
TABLE = "IOT"
YEARS = range(2013, 2015)
SYSTEM = "pxp"
SECTOR = "Nickel ores and concentrates"

# Uncomment if you need to download the dataset first.
# mario.download_exiobase(version="3.10.1", system=SYSTEM, years=YEARS, path=FOLDER)

## 2. Parse the first year and import the rest as scenarios

This keeps the whole time series inside one `Database`. That matters because all the supply-chain indicators below can then be compared with `scenario="all"` or by selecting one baseline year.

In [25]:
db = mario.parse_exiobase(
    path=f"{FOLDER}/{TABLE}_{YEARS[0]}_{SYSTEM}.zip",
    table="IOT",
    unit="Monetary",
)
db.rename_scenario("baseline", YEARS[0])

INFO Parser: reading EXIOBASE IOT from /var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/mario_exiobase_iot_aacpo95a.
INFO Parser: reading EXIOBASE IOT extensions from /var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/mario_exiobase_iot_aacpo95a.
INFO Parser: using split extension layout with 7 extension directories.
INFO Parser: EXIOBASE IOT parsed with 200 sectors, 9 value-added rows and 725 extension rows.
INFO Metadata: initialized.


In [ ]:
for year in YEARS[1:]:
    print(f"Processing year {year}...")
    db.parse_exiobase(
        path=f"{FOLDER}/{TABLE}_{year}_{SYSTEM}.zip",
        table="IOT",
        unit="Monetary",
        new_scenario=year,
    )
    print("DONE")

Processing year 2014...
INFO Parser: reading EXIOBASE IOT from /var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/mario_exiobase_iot_176gbwyq.
INFO Parser: reading EXIOBASE IOT extensions from /var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/mario_exiobase_iot_176gbwyq.
INFO Parser: using split extension layout with 7 extension directories.
INFO Parser: EXIOBASE IOT parsed with 200 sectors, 9 value-added rows and 725 extension rows.
INFO Parser: state payload ready with 5 canonical blocks.
DONE


## 3. Start from structural position: `calc_linkages(...)`

A complete supply-chain analysis should begin with the sector's structural role in the production network. `calc_linkages(...)` tells you whether the focal sector is mainly a strong demander of upstream inputs, a strong downstream enabler, or both.

In multi-regional mode, MARIO also reports `Local Share` and `Foreign Share`, which help separate linkage intensity from linkage geography. In aggregated mode, it adds amplification ratios that show whether indirect effects are much larger than the direct ones.

In [27]:
linkages = db.calc_linkages(scenario="2013")
linkages.head(10)

INFO Resolver: resolving w for 2013.
INFO Resolver: trying w via formula build_iot_w_from_z.
INFO Resolver: resolved w via formula build_iot_w_from_z.
INFO Resolver: resolving b for 2013.
INFO Resolver: trying b via formula build_iot_b_from_X_Z.
INFO Resolver: resolved b via formula build_iot_b_from_X_Z.
INFO Resolver: resolving g for 2013.
INFO Resolver: trying g via formula build_iot_g_from_b.
INFO Resolver: resolved g via formula build_iot_g_from_b.
WARNING Normalization not available for multi-regional mode.


Total Forward  \
                                                                         Local   
Region Level  Item                                                               
AT     Sector Additives/Blending Components                           0.000000   
              Air transport services (62)                             0.354104   
              Aluminium and aluminium products                        0.696550   
              Aluminium ores and concentrates                         0.000000   
              Animal products nec                                     0.098513   
              Anthracite                                              0.029730   
              Ash for treatment, Re-processing of ash into cl...      0.000000   
              Aviation Gasoline                                       0.536250   
              BKB/Peat Briquettes                                     0.000000   
              Basic iron and steel and of ferro-alloys and fi...      0.042204   

                                                                            \
                                                                   Foreign   
Region Level  Item                                                           
AT     Sector Additives/Blending Components                       0.000000   
              Air transport services (62)                         0.390661   
              Aluminium and aluminium products                    1.513491   
              Aluminium ores and concentrates                     0.000000   
              Animal products nec                                 0.432467   
              Anthracite                                          2.908958   
              Ash for treatment, Re-processing of ash into cl...  0.000000   
              Aviation Gasoline                                   1.253700   
              BKB/Peat Briquettes                                 0.000000   
              Basic iron and steel and of ferro-alloys and fi...  2.440849   

                                                                 Total Backward  \
                                                                          Local   
Region Level  Item                                                                
AT     Sector Additives/Blending Components                            0.000000   
              Air transport services (62)                              0.408775   
              Aluminium and aluminium products                         0.781241   
              Aluminium ores and concentrates                          0.000000   
              Animal products nec                                      0.668477   
              Anthracite                                               1.001713   
              Ash for treatment, Re-processing of ash into cl...       0.000000   
              Aviation Gasoline                                        0.379844   
              BKB/Peat Briquettes                                      0.000000   
              Basic iron and steel and of ferro-alloys and fi...       0.708880   

                                                                            \
                                                                   Foreign   
Region Level  Item                                                           
AT     Sector Additives/Blending Components                       0.000000   
              Air transport services (62)                         0.817409   
              Aluminium and aluminium products                    1.069240   
              Aluminium ores and concentrates                     0.000000   
              Animal products nec                                 0.573460   
              Anthracite                                          0.605283   
              Ash for treatment, Re-processing of ash into cl...  0.000000   
              Aviation Gasoline                                   1.128521   
              BKB/Peat Briquettes            

In [28]:
aggregated_linkages = db.calc_linkages(
    scenario="2013",
    multi_mode=False,
 )

aggregated_linkages.head(10)

Total Forward  \
Region Level  Item                                                                
AT     Sector Additives/Blending Components                            0.000000   
              Air transport services (62)                              0.602529   
              Aluminium and aluminium products                         1.787964   
              Aluminium ores and concentrates                          0.000000   
              Animal products nec                                      0.429572   
              Anthracite                                               2.377452   
              Ash for treatment, Re-processing of ash into cl...       0.000000   
              Aviation Gasoline                                        1.448102   
              BKB/Peat Briquettes                                      0.000000   
              Basic iron and steel and of ferro-alloys and fi...       2.008835   

                                                                  Total Backward  \
Region Level  Item                                                                 
AT     Sector Additives/Blending Components                             0.000000   
              Air transport services (62)                               1.219589   
              Aluminium and aluminium products                          1.840527   
              Aluminium ores and concentrates                           0.000000   
              Animal products nec                                       1.235257   
              Anthracite                                                1.598352   
              Ash for treatment, Re-processing of ash into cl...        0.000000   
              Aviation Gasoline                                         1.500252   
              BKB/Peat Briquettes                                       0.000000   
              Basic iron and steel and of ferro-alloys and fi...        1.548759   

                                                                  Direct Forward  \
Region Level  Item                                                                 
AT     Sector Additives/Blending Components                             0.000000   
              Air transport services (62)                               0.574110   
              Aluminium and aluminium products                          1.803558   
              Aluminium ores and concentrates                           0.000000   
              Animal products nec                                       0.512181   
              Anthracite                                                1.835213   
              Ash for treatment, Re-processing of ash into cl...        0.000000   
              Aviation Gasoline                                         1.846841   
              BKB/Peat Briquettes                                       0.000000   
              Basic iron and steel and of ferro-alloys and fi...        1.881335   

                                                                  Direct Backward  \
Region Level  Item                                                                  
AT     Sector Additives/Blending Components                              0.000000   
              Air transport services (62)                                1.140037   
              Aluminium and aluminium products                           1.571110   
              Aluminium ores and concentrates                            0.000000   
              Animal products nec                                        1.169562   
              Anthracite                                                 1.582155   
              Ash for treatment, Re-processing of ash into cl...         0.000000   
              Aviation Gasoline                                          1.706059   
              BKB/Peat Briquettes                                        0.000000   
              Basic iron and steel and of ferro-alloys and fi...         1.455337   

                              

## 4. Map the bilateral geography of the chain with `calc_trades(...)`

Once you know whether the sector is structurally important, the next question is geographic: where do the relevant trade flows actually run? `calc_trades(...)` answers that by building origin-by-destination trade matrices.

The examples below keep the split between intermediate and final trade, show totals, and then aggregate the origin side to continents. The same method can also build an animated plot with `scenario="all"`.

In [29]:
split_trade = db.calc_trades(
    item=SECTOR,
    scenario="2013",
    aggregate=False,
    total=True,
    auto_open=False,
    path=False,
    show_plot=False,
)

split_trade.head(10)

Intermediate                                                            \
Region           AT           AU        BE   BG          BR           CA   CH   
Region                                                                          
AT              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
AU              0.0  2096.333717  0.008412  0.0   42.148222    71.910781  0.0   
BE              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
BG              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
BR              0.0     0.000000  0.087826  0.0  576.120518     0.118718  0.0   
CA              0.0     0.000000  0.000000  0.0    0.000000  1105.895311  0.0   
CH              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
CN              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
CY              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   
CZ              0.0     0.000000  0.000000  0.0    0.000000     0.000000  0.0   

                                    ... Final                               \
Region           CN   CY        CZ  ...    TR   TW            US   WA   WE   
Region                              ...                                      
AT         0.000000  0.0  0.000000  ...   0.0  0.0  0.000000e+00  0.0  0.0   
AU      1378.824431  0.0  0.000000  ...   0.0  0.0  0.000000e+00  0.0  0.0   
BE         0.000000  0.0  0.000000  ...   0.0  0.0  0.000000e+00  0.0  0.0   
BG         0.000000  0.0  0.000000  ...   0.0  0.0  0.000000e+00  0.0  0.0   
BR         4.715478  0.0  0.000000  ...   0.0  0.0  0.000000e+00  0.0  0.0   
CA         1.166070  0.0  0.000000  ...   0.0  0.0  0.000000e+00  0.0  0.0   
CH         0.000000  0.0  0.000000  ...   0.0  0.0  0.000000e+00  0.0  0.0   
CN      2374.228873  0.0  0.000000  ...   0.0  0.0  3.310822e-14  0.0  0.0   
CY         0.000000  0.0  0.000000  ...   0.0  0.0  0.000000e+00  0.0  0.0   
CZ         0.000000  0.0  0.008093  ...   0.0  0.0  0.000000e+00  0.0  0.0   

                                            
Region   WF   WL        WM   ZA      Total  
Region                                      
AT      0.0  0.0  0.000000  0.0   0.000000  
AU      0.0  0.0  0.011084  0.0  14.873840  
BE      0.0  0.0  0.000000  0.0   0.000000  
BG      0.0  0.0  0.000000  0.0   0.000000  
BR      0.0  0.0  0.000000  0.0  16.708715  
CA      0.0  0.0  0.027729  0.0   0.756920  
CH      0.0  0.0  0.000000  0.0   0.000000  
CN      0.0  0.0  0.000000  0.0   0.211057  
CY      0.0  0.0  0.000000  0.0   0.000000  
CZ      0.0  0.0  0.000000  0.0   0.147689  

[10 rows x 100 columns]

In [31]:
origin_clustered_trades = db.calc_trades(
    item=SECTOR,
    scenario="2013",
    clusters="continent",
    clusters_direction="origin",
    total=True,
    auto_open=False,
    path=False,
    show_plot=False,
)
origin_clustered_trades

Region,AT,AU,BE,BG,BR,CA,CH,CN,CY,CZ,...,TR,TW,US,WA,WE,WF,WL,WM,ZA,Total
Region,,,,,,,,,,,,,,,,,,,,,
Europe,0.0,0.000000,0.908103,0.0,0.000000,51.909231,0.0,160.407913,9.757916e-13,1.659926,...,0.002653,0.0,0.002786,0.072842,132.738378,15.831240,0.000000,0.000000,4.469047,2236.514003
Asia,0.0,70.382138,0.018823,0.0,0.000000,0.000000,0.0,5673.822022,0.000000e+00,0.000000,...,26.589091,0.0,71.057130,841.185627,52.852502,0.003665,4.146893,236.196678,0.000000,9182.657505
America,0.0,0.000000,0.087826,0.0,592.822909,1106.740844,0.0,5.881573,0.000000e+00,0.000000,...,0.000000,0.0,0.006253,0.000000,56.187196,0.000000,318.642076,2.372726,0.000000,2156.621242
Oceania,0.0,2097.503758,0.008412,0.0,55.834461,71.910781,0.0,1378.830230,0.000000e+00,0.000000,...,0.000000,0.0,0.000000,0.015612,0.000000,0.000000,0.000000,0.948428,0.000000,3639.674247
Africa,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,21.008060,0.000000e+00,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,1089.456976,0.000000,0.000000,448.475774,1581.516306
Total,0.0,2167.885896,1.023165,0.0,648.657370,1230.560856,0.0,7239.949798,9.757916e-13,1.659926,...,26.591744,0.0,71.066169,841.274081,241.778076,1105.291881,322.788969,239.517833,452.944821,18796.983303


In [32]:
all_trades = db.calc_trades(
    item=SECTOR,
    scenario="all",
    show_plot=True,
    auto_open=False,
    path=False,
    title=f"{SECTOR} - all scenarios",
)

If you want a shareable figure, the same method can also export HTML or static images.

```python
db.calc_trades(
    item=SECTOR,
    scenario=base_scenario,
    save_plot=f"{FOLDER}/nickel_trade_{base_label}.html",
    auto_open=False,
 )
```

In practice this block answers the first geographic question of the analysis: which origin-destination pairs actually carry the focal supply chain?

## 5. Add embodied indicators with `calc_trades_content(...)`

Trade values alone are not enough if the question is about value added, emissions, energy, water, labour, or other extensions. `calc_trades_content(...)` lets you attach those indicators directly to the same trade geometry.

Here `indicator` means the quantity embodied in the traded flow. It can be one factor-of-production row, one satellite-account row, or the generic `"total value added"` shortcut.

The `method` argument changes the analytical meaning of the result:

- `direct`: only the content generated directly by the production node that supplies the traded item
- `total`: direct content plus all indirect upstream rounds required to produce that traded item
- `upstream`: only the indirect upstream contribution, so it isolates supplier-chain dependence net of the direct producer itself

A useful way to read them is: if `total` is much larger than `direct`, the traded flow carries an important hidden upstream chain. If `upstream` is large, the indicator is not mainly generated by the last exporting sector, but by its wider supplier network.

The first example below uses one factor-of-production row and focuses on the `upstream` part of the chain.

In [33]:
factor_indicator = db.get_index("Factor of production")[0]

upstream_value_added = db.calc_trades_content(
    indicator=factor_indicator,
    item=SECTOR,
    scenario="2013",
    method="upstream",
    clusters="continent",
    clusters_direction="origin",
    show_plot=True,
    auto_open=False,
    path=False,
    title=f"{factor_indicator} embodied in {SECTOR} trade",
)
upstream_value_added

INFO Resolver: resolving v for 2013.
INFO Resolver: trying v via formula build_iot_v_from_V_X.
INFO Resolver: resolved v via formula build_iot_v_from_V_X.
INFO Resolver: resolving m for 2013.
INFO Resolver: trying m via formula build_iot_m_from_v_w (compute_method=auto, runtime=inverse).
INFO Resolver: resolved m via formula build_iot_m_from_v_w (compute_method=auto, runtime=inverse).


Region,AT,AU,BE,BG,BR,CA,CH,CN,CY,CZ,...,SK,TR,TW,US,WA,WE,WF,WL,WM,ZA
Region,,,,,,,,,,,,,,,,,,,,,
Europe,0.0,0.000000,0.051146,0.0,0.000000,1.705098,0.0,3.271703,1.128445e-14,0.018337,...,0.000381,0.000031,0.0,0.000053,0.000965,1.535070,0.520025,0.000000,0.000000,0.146799
Asia,0.0,0.470155,0.000102,0.0,0.000000,0.000000,0.0,69.824418,0.000000e+00,0.000000,...,0.000000,0.594932,0.0,0.384311,4.671407,0.657557,0.000020,0.050877,2.917291,0.000000
America,0.0,0.000000,0.006531,0.0,44.081402,19.193387,0.0,0.370852,0.000000e+00,0.000000,...,0.000000,0.000000,0.0,0.000040,0.000000,0.359475,0.000000,2.039046,0.041134,0.000000
Oceania,0.0,21.327568,0.000086,0.0,0.567729,0.731194,0.0,14.020044,0.000000e+00,0.000000,...,0.000000,0.000000,0.0,0.000000,0.000159,0.000000,0.000000,0.000000,0.009644,0.000000
Africa,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.171755,0.000000e+00,0.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,8.950772,0.000000,0.000000,5.049428


In [35]:
all_value_added_content = db.calc_trades_content(
    indicator="total value added",
    item=SECTOR,
    scenario="2013",
    method="total",
    show_plot=True,
    auto_open=False,
    path=False,
    title=f"Total value added embodied in {SECTOR} trade",
)
all_value_added_content.head(10)

Region,AT,AU,BE,BG,BR,CA,CH,CN,CY,CZ,...,SK,TR,TW,US,WA,WE,WF,WL,WM,ZA
Region,,,,,,,,,,,,,,,,,,,,,
AT,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0
AU,0.0,2097.503759,0.008412,0.0,55.834461,71.910781,0.0,1378.830231,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.015612,0.0,0.0,0.000000,0.948428,0.0
BE,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0
BG,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0
BR,0.0,0.000000,0.087826,0.0,592.822909,0.118718,0.0,4.715497,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.005586,0.000000,0.0
CA,0.0,0.000000,0.000000,0.0,0.000000,1106.622126,0.0,1.166075,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.004880,2.372726,0.0
CH,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0
CN,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,2374.439931,0.0,0.000000,...,0.000000,0.0,0.0,0.002177,0.135476,0.0,0.0,2.401789,0.000000,0.0
CY,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0


## 6. Explain the result with `breakdown=True`, concentration, and exposure

A total embodied-content matrix tells you how much indicator content is carried by each trade pair, but not where that content originates inside the production system. `breakdown=True` keeps the contributor detail so you can identify the production nodes behind the result.

In practice, the breakdown table answers: `who is actually generating the embodied content behind this observed trade flow?` The first row level still identifies the trade origin Region, while the deeper levels identify the contributing production nodes. If you sum those contributor levels back together, you recover the collapsed trade-content matrix for the same method.

The last two indicators are summaries built on top of that contributor table, and they answer different questions:

- `calc_trades_concentration(...)`: how concentrated is the contributor base behind each trade pair?
- `calc_trades_exposure(...)`: how much of that contributor base depends on one selected Region or group of Regions?

`calc_trades_concentration(...)` is computed as an HHI on contributor-Region shares. A value closer to 1 means that only a few contributor Regions dominate the embodied content. Lower values mean the same content is spread across a broader set of contributor Regions. This is a diversification indicator, not a size indicator: a small trade flow and a large trade flow can have the same concentration if their contributor shares are distributed in the same way.

`calc_trades_exposure(...)` instead returns a share between 0 and 1. It measures how much of the embodied content is attributable to the Regions selected in `exposed_to`. High exposure means strong dependence on those Regions even if the observed exporter-importer map looks diversified. In other words, direct trade partners and hidden upstream dependency are not necessarily the same thing.

In [36]:
value_added_breakdown = db.calc_trades_content(
    indicator="total value added",
    item=SECTOR,
    scenario="2013",
    method="upstream",
    breakdown=True,
    clusters="continent",
    clusters_direction="origin",
)

value_added_breakdown.head(10)

Region                                                                    AT  \
Region Region Level  Item                                                      
Europe AT     Sector Additives/Blending Components                       0.0   
                     Air transport services (62)                         0.0   
                     Aluminium and aluminium products                    0.0   
                     Aluminium ores and concentrates                     0.0   
                     Animal products nec                                 0.0   
                     Anthracite                                          0.0   
                     Ash for treatment, Re-processing of ash into cl...  0.0   
                     Aviation Gasoline                                   0.0   
                     BKB/Peat Briquettes                                 0.0   
                     Basic iron and steel and of ferro-alloys and fi...  0.0   

Region                                                                    AU  \
Region Region Level  Item                                                      
Europe AT     Sector Additives/Blending Components                       0.0   
                     Air transport services (62)                         0.0   
                     Aluminium and aluminium products                    0.0   
                     Aluminium ores and concentrates                     0.0   
                     Animal products nec                                 0.0   
                     Anthracite                                          0.0   
                     Ash for treatment, Re-processing of ash into cl...  0.0   
                     Aviation Gasoline                                   0.0   
                     BKB/Peat Briquettes                                 0.0   
                     Basic iron and steel and of ferro-alloys and fi...  0.0   

Region                                                                             BE  \
Region Region Level  Item                                                               
Europe AT     Sector Additives/Blending Components                       0.000000e+00   
                     Air transport services (62)                         1.370348e-05   
                     Aluminium and aluminium products                    1.291321e-05   
                     Aluminium ores and concentrates                     0.000000e+00   
                     Animal products nec                                 4.824001e-08   
                     Anthracite                                          8.834852e-10   
                     Ash for treatment, Re-processing of ash into cl...  0.000000e+00   
                     Aviation Gasoline                                   2.051568e-09   
                     BKB/Peat Briquettes                                 0.000000e+00   
                     Basic iron and steel and of ferro-alloys and fi...  4.762819e-05   

Region                                                                    BG  \
Region Region Level  Item                                                      
Europe AT     Sector Additives/Blending Components                       0.0   
                     Air transport services (62)                         0.0   
                     Aluminium and aluminium products                    0.0   
                     Aluminium ores and concentrates                     0.0   
                     Animal products nec                                 0.0   
                     Anthracite                                          0.0   
                     Ash for treatment, Re-processing of ash into cl...  0.0   
                     Aviation Gasoline                                   0.0   
                     BKB/Peat Briquettes                                 0.0   
                     Basic iron and steel and of ferro-alloys and fi...  0.0   

Region                                          

In [37]:
value_added_concentration = db.calc_trades_concentration(
    indicator="total value added",
    item=SECTOR,
    scenario="2013",
    method="upstream",
    clusters="continent",
    clusters_direction="origin",
)

value_added_concentration

Region,AT,AU,BE,BG,BR,CA,CH,CN,CY,CZ,...,SK,TR,TW,US,WA,WE,WF,WL,WM,ZA
Region,,,,,,,,,,,,,,,,,,,,,
Europe,0.0,0.000000,0.669498,0.0,0.000000,0.218142,0.0,0.225551,0.311919,0.114916,...,0.21443,0.311919,0.0,0.410742,0.380046,0.311915,0.218150,0.000000,0.000000,0.218150
Oceania,0.0,0.449936,0.449936,0.0,0.449936,0.449936,0.0,0.449936,0.000000,0.000000,...,0.00000,0.000000,0.0,0.000000,0.449936,0.000000,0.000000,0.000000,0.449936,0.000000
America,0.0,0.000000,0.516330,0.0,0.516330,0.532473,0.0,0.318597,0.000000,0.000000,...,0.00000,0.000000,0.0,0.351588,0.000000,0.351588,0.000000,0.351566,0.532548,0.000000
Asia,0.0,0.176630,0.288928,0.0,0.000000,0.000000,0.0,0.203990,0.000000,0.000000,...,0.00000,0.218057,0.0,0.288905,0.267167,0.227852,0.288928,0.375378,0.292964,0.000000
Africa,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.330962,0.000000,0.000000,...,0.00000,0.000000,0.0,0.000000,0.000000,0.000000,0.321583,0.000000,0.000000,0.336972


In [20]:
value_added_exposure = db.calc_trades_exposure(
    indicator="total value added",
    exposed_to=focus_regions,
    item=SECTOR,
    scenario=base_scenario,
    method="upstream",
    clusters="continent",
    clusters_direction="origin",
)

value_added_exposure

Region,AT,AU,BE,BG,BR,CA,CH,CN,CY,CZ,...,SK,TR,TW,US,WA,WE,WF,WL,WM,ZA
Region,,,,,,,,,,,,,,,,,,,,,
Europe,0.0,0.000000,0.006872,0.0,0.000000,0.013652,0.0,0.013147,0.013172,0.010972,...,0.02102,0.013172,0.0,0.013260,0.010501,0.013172,0.013653,0.000000,0.000000,0.013653
Oceania,0.0,0.006377,0.006377,0.0,0.006377,0.006377,0.0,0.006377,0.000000,0.000000,...,0.00000,0.000000,0.0,0.000000,0.006377,0.000000,0.000000,0.000000,0.006377,0.000000
America,0.0,0.000000,0.003705,0.0,0.003705,0.005338,0.0,0.004143,0.000000,0.000000,...,0.00000,0.000000,0.0,0.004968,0.000000,0.004968,0.000000,0.004968,0.005338,0.000000
Asia,0.0,0.005992,0.004668,0.0,0.000000,0.000000,0.0,0.006026,0.000000,0.000000,...,0.00000,0.014183,0.0,0.004668,0.004845,0.008383,0.004668,0.004729,0.011143,0.000000
Africa,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.011062,0.000000,0.000000,...,0.00000,0.000000,0.0,0.000000,0.000000,0.000000,0.010961,0.000000,0.000000,0.005592


Read the last two outputs together, not in isolation:

- high concentration and high exposure: the embodied chain is dominated by a narrow set of contributors, and the selected Regions are a large part of that bottleneck
- high concentration and low exposure: the chain is still concentrated, but the dominant contributors are mostly outside the Regions selected in `exposed_to`
- low concentration and high exposure: the selected Regions matter a lot, but dependence is spread across several contributors rather than one single bottleneck
- low concentration and low exposure: the chain is relatively diversified and not strongly tied to the selected Regions

So concentration is a question about the structure of the whole contributor base, while exposure is a question about dependence on a specific target.

## 7. Repeat the same logic for satellite accounts

The same workflow can be reused for environmental extensions. First inspect the available labels, then choose the indicator that matches the footprint you want to track.

In [38]:
db.search("CO2")

{'Satellite account': ['CO2 - combustion - air',
  'CO2_bio - combustion - air',
  'CO2 - non combustion - Cement production - air',
  'CO2 - non combustion - Lime production - air',
  'CO2 - agriculture - peat decay - air',
  'CO2 - waste - biogenic - air',
  'CO2 - waste - fossil - air']}

In [39]:
CO2_INDICATOR = "CO2 - combustion - air" # select one result of the search above

co2_upstream_content = db.calc_trades_content(
    indicator=CO2_INDICATOR,
    item=SECTOR,
    scenario=base_scenario,
    method="upstream",
    show_plot=True,
    auto_open=False,
    path=False,
    title=f"{CO2_INDICATOR} embodied in {SECTOR} trade",
)
co2_upstream_content.head(10)

INFO Resolver: resolving e for 2013.
INFO Resolver: trying e via formula build_iot_e_from_E_X.
INFO Resolver: resolved e via formula build_iot_e_from_E_X.
INFO Resolver: resolving f for 2013.
INFO Resolver: trying f via formula build_iot_f_from_e_w (compute_method=auto, runtime=inverse).
INFO Resolver: resolved f via formula build_iot_f_from_e_w (compute_method=auto, runtime=inverse).


Region,AT,AU,BE,BG,BR,CA,CH,CN,CY,CZ,...,SK,TR,TW,US,WA,WE,WF,WL,WM,ZA
Region,,,,,,,,,,,,,,,,,,,,,
AT,0.0,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,0.0
AU,0.0,1.215395e+09,4874.435708,0.0,3.235317e+07,4.166857e+07,0.0,7.989606e+08,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,9046.622707,0.0,0.0,0.000000e+00,5.495651e+05,0.0
BE,0.0,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,0.0
BG,0.0,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,0.0
BR,0.0,0.000000e+00,18845.824215,0.0,1.272085e+08,2.547471e+04,0.0,1.011856e+06,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,1.198624e+03,0.000000e+00,0.0
CA,0.0,0.000000e+00,0.000000,0.0,0.000000e+00,4.993165e+08,0.0,5.261423e+05,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,2.202058e+03,1.070593e+06,0.0
CH,0.0,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,0.0
CN,0.0,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,7.642919e+09,0.0,0.000000,...,0.000000,0.0,0.0,7005.962304,436075.119212,0.0,0.0,7.730950e+06,0.000000e+00,0.0
CY,0.0,0.000000e+00,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,0.0


## 8. Read the workflow as one complete supply-chain analysis

A practical full-chain reading usually combines the methods above in this order:

1. `calc_linkages(...)`: is the focal sector structurally important upstream, downstream, or both?
2. `calc_trades(...)`: where are the relevant bilateral trade flows located?
3. `calc_trades_content(...)`: how much value added or environmental pressure is embodied in those flows, and how much comes from direct production versus the wider supplier chain?
4. `breakdown=True`: which contributors actually generate that embodied content behind each trade pair?
5. `calc_trades_concentration(...)`: is that embodied content diversified across many contributor Regions or dominated by a few bottlenecks?
6. `calc_trades_exposure(...)`: what share of that embodied content depends on a specific Region or group of Regions?

That sequence turns MARIO from a parser into a full supply-chain analysis workflow: structure, geography, embodied indicators, hidden contributors, concentration, and dependence inside one EXIOBASE-backed database.